# Эксперименты с моделями

Ноутбук состоит из двух частей:

- Часть 1 - сравнение 4-5 алгоритмов с дефолтными параметрами (без дополнительного feature engineering).
- Часть 2 - тюнинг гиперпараметров лучших моделей, эксперименты с PCA, итоговая таблица экспериментов.

По итогам выбирается финальная модель и обосновывается выбор.

In [ ]:
import sys
import warnings

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

sys.path.append("../src")
from preprocessing import (
    CAT_COLS,
    NUM_COLS,
    TARGET,
    clean_data,
    engineer_features,
    load_raw_data,
    split_data,
)
from train import build_pipeline, evaluate, save_model, results_table

warnings.filterwarnings("ignore")

SEED = 42
DATA_PATH = "../data/raw/adult.csv"
BEST_MODEL_PATH = "../models/best_model.joblib"

plt.rcParams["figure.dpi"] = 120

## Загрузка и подготовка данных

In [ ]:
df_raw = load_raw_data(DATA_PATH)
df = clean_data(df_raw)

X = df[CAT_COLS + NUM_COLS]
y = df[TARGET]

X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y, seed=SEED)

print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

---
# Часть 1 - Сравнение моделей с дефолтными параметрами

Каждая модель обёрнута в пайплайн `OneHotEncoder + StandardScaler -> model`.  
Feature engineering не применяется, используются только исходные 14 признаков.  
Цель - получить базовую картину и выбрать кандидатов для тюнинга.

In [ ]:
CANDIDATES = {
    "DecisionTree": DecisionTreeClassifier(random_state=SEED),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=100, random_state=SEED),
    "XGBoost": XGBClassifier(
        n_estimators=100, random_state=SEED, eval_metric="logloss",
        use_label_encoder=False, n_jobs=-1,
    ),
    "LightGBM": LGBMClassifier(n_estimators=100, random_state=SEED, n_jobs=-1, verbose=-1),
}

In [ ]:
part1_rows = []

for name, model in CANDIDATES.items():
    pipe = build_pipeline(model)
    pipe.fit(X_train, y_train)
    metrics = evaluate(pipe, X_val, y_val, split_name="val")
    part1_rows.append({"model": name, **{k: v for k, v in metrics.items() if k != "split"}})
    print(f"{name:<20}  ROC-AUC: {metrics['roc_auc']:.4f}  F1-macro: {metrics['f1_macro']:.4f}")

In [ ]:
df_part1 = pd.DataFrame(part1_rows).set_index("model").sort_values("roc_auc", ascending=False)
df_part1

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col in zip(axes, ["roc_auc", "f1_macro"]):
    df_part1[col].sort_values().plot(kind="barh", ax=ax)
    ax.set_title(col)
    ax.set_xlabel(col)
plt.suptitle("Часть 1: сравнение моделей (дефолтные параметры, val)")
plt.tight_layout()
plt.show()

---
# Часть 2 - Тюнинг гиперпараметров и эксперименты

## 2.1 Эксперимент: уменьшение размерности (PCA)

Гипотеза: после OneHotEncoding признаковое пространство сильно расширяется.
PCA может снизить шум и ускорить обучение без существенной потери качества.

Как проверяли: обучаем RandomForest через `build_pipeline(use_pca=True, n_components=k)` при k = 10, 20, 30 и без PCA.

In [ ]:
pca_rows = []
for n_comp in [10, 20, 30, None]:
    use_pca = n_comp is not None
    pipe = build_pipeline(
        RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
        use_pca=use_pca,
        n_components=n_comp if use_pca else 10,
    )
    pipe.fit(X_train, y_train)
    metrics = evaluate(pipe, X_val, y_val, split_name="val")
    label = f"RF + PCA({n_comp})" if use_pca else "RF (no PCA)"
    pca_rows.append({"model": label, **{k: v for k, v in metrics.items() if k != "split"}})
    print(f"{label:<22}  ROC-AUC: {metrics['roc_auc']:.4f}  F1-macro: {metrics['f1_macro']:.4f}")

pd.DataFrame(pca_rows).set_index("model")

## 2.2 Тюнинг RandomForest (RandomizedSearchCV)

Гипотеза: подбор `n_estimators`, `max_depth`, `min_samples_leaf` и `max_features` улучшит ROC-AUC RandomForest.

In [ ]:
rf_param_dist = {
    "model__n_estimators": randint(100, 400),
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_leaf": randint(1, 10),
    "model__max_features": ["sqrt", "log2", 0.5],
    "model__class_weight": [None, "balanced"],
}

rf_base = build_pipeline(RandomForestClassifier(random_state=SEED, n_jobs=-1))

rf_search = RandomizedSearchCV(
    rf_base,
    param_distributions=rf_param_dist,
    n_iter=20,
    scoring="roc_auc",
    cv=3,
    random_state=SEED,
    n_jobs=-1,
    refit=True,
)
rf_search.fit(X_train, y_train)

rf_best = rf_search.best_estimator_
rf_metrics = evaluate(rf_best, X_val, y_val, split_name="val")
print("Best params:", rf_search.best_params_)
print(f"RF tuned — ROC-AUC: {rf_metrics['roc_auc']:.4f}  F1-macro: {rf_metrics['f1_macro']:.4f}")

## 2.3 Тюнинг LightGBM (RandomizedSearchCV)

Гипотеза: LightGBM с оптимальными `num_leaves`, `learning_rate`, `n_estimators` и `reg_lambda` покажет лучший ROC-AUC, чем дефолт.

In [ ]:
lgbm_param_dist = {
    "model__n_estimators": randint(100, 500),
    "model__num_leaves": randint(20, 100),
    "model__learning_rate": uniform(0.01, 0.2),
    "model__min_child_samples": randint(10, 50),
    "model__reg_lambda": uniform(0.0, 1.0),
    "model__class_weight": [None, "balanced"],
}

lgbm_base = build_pipeline(LGBMClassifier(random_state=SEED, n_jobs=-1, verbose=-1))

lgbm_search = RandomizedSearchCV(
    lgbm_base,
    param_distributions=lgbm_param_dist,
    n_iter=20,
    scoring="roc_auc",
    cv=3,
    random_state=SEED,
    n_jobs=-1,
    refit=True,
)
lgbm_search.fit(X_train, y_train)

lgbm_best = lgbm_search.best_estimator_
lgbm_metrics = evaluate(lgbm_best, X_val, y_val, split_name="val")
print("Best params:", lgbm_search.best_params_)
print(f"LightGBM tuned — ROC-AUC: {lgbm_metrics['roc_auc']:.4f}  F1-macro: {lgbm_metrics['f1_macro']:.4f}")

## 2.4 Эксперимент: feature engineering

Гипотеза: добавление новых признаков (`capital_net`, `age_group`, `is_usa`) из EDA улучшит качество лучшей модели.

Как проверяли: обучаем LightGBM (дефолт и с параметрами из тюнинга) на расширенном наборе признаков, сравниваем с версией без FE.

In [ ]:
FE_CAT_COLS = CAT_COLS + ["age_group"]
FE_NUM_COLS = NUM_COLS + ["capital_net", "is_usa"]

df_fe = engineer_features(df)
X_fe = df_fe[FE_CAT_COLS + FE_NUM_COLS]
X_train_fe, X_val_fe, X_test_fe, y_train_fe, y_val_fe, y_test_fe = split_data(X_fe, y, seed=SEED)

lgbm_fe_default = build_pipeline(
    LGBMClassifier(n_estimators=100, random_state=SEED, n_jobs=-1, verbose=-1),
    cat_cols=FE_CAT_COLS,
    num_cols=FE_NUM_COLS,
)
lgbm_fe_default.fit(X_train_fe, y_train_fe)
fe_default_m = evaluate(lgbm_fe_default, X_val_fe, y_val_fe, split_name="val")
print(f"LightGBM + FE (дефолт)  — ROC-AUC: {fe_default_m['roc_auc']:.4f}  F1-macro: {fe_default_m['f1_macro']:.4f}")

best_lgbm_params = {k.replace("model__", ""): v for k, v in lgbm_search.best_params_.items()}
lgbm_fe_tuned = build_pipeline(
    LGBMClassifier(**best_lgbm_params, random_state=SEED, n_jobs=-1, verbose=-1),
    cat_cols=FE_CAT_COLS,
    num_cols=FE_NUM_COLS,
)
lgbm_fe_tuned.fit(X_train_fe, y_train_fe)
fe_tuned_m = evaluate(lgbm_fe_tuned, X_val_fe, y_val_fe, split_name="val")
print(f"LightGBM + FE (тюнинг)  — ROC-AUC: {fe_tuned_m['roc_auc']:.4f}  F1-macro: {fe_tuned_m['f1_macro']:.4f}")

## 2.5 Эксперимент: ансамбль (VotingClassifier)

Гипотеза: объединение прогнозов нескольких разных моделей через мягкое голосование снизит дисперсию ошибки и даст прирост ROC-AUC.

Как проверяли: `VotingClassifier(voting='soft')` с тремя лучшими моделями из части 1: LightGBM (с параметрами из тюнинга), XGBoost, GradientBoosting.

In [ ]:
from sklearn.ensemble import VotingClassifier

best_lgbm_params = {k.replace("model__", ""): v for k, v in lgbm_search.best_params_.items()}

pipe_lgbm_v = build_pipeline(LGBMClassifier(**best_lgbm_params, random_state=SEED, verbose=-1))
pipe_xgb_v = build_pipeline(
    XGBClassifier(n_estimators=100, random_state=SEED, eval_metric="logloss",
                  use_label_encoder=False)
)
pipe_gb_v = build_pipeline(GradientBoostingClassifier(n_estimators=100, random_state=SEED))

ensemble = VotingClassifier(
    estimators=[("lgbm", pipe_lgbm_v), ("xgb", pipe_xgb_v), ("gb", pipe_gb_v)],
    voting="soft",
)
ensemble.fit(X_train, y_train)
ensemble_m = evaluate(ensemble, X_val, y_val, split_name="val")
print(f"VotingClassifier (LGBM+XGB+GB) — ROC-AUC: {ensemble_m['roc_auc']:.4f}  F1-macro: {ensemble_m['f1_macro']:.4f}")

## 2.6 Сводная таблица всех экспериментов

| # | Гипотеза | Как проверялось | Лучший ROC-AUC (val) | F1-macro (val) |
|---|---|---|---|---|
| 1 | Дефолтные алгоритмы, нет разницы между ними | Обучение 5 моделей без тюнинга | 0.8837 (LightGBM) | 0.6980 |
| 2 | PCA снизит шум и улучшит RF | RF + PCA(10/20/30) vs RF без PCA | 0.8536 (PCA=10) vs 0.8535 (без) | 0.6544 vs 0.6716 |
| 3 | Тюнинг RF даст прирост | RandomizedSearchCV (20 iter, cv=3) | 0.8766 (+0.023 к дефолту) | 0.6596 |
| 4 | LightGBM с тюнингом - лучшая одиночная модель | RandomizedSearchCV (20 iter, cv=3) | 0.8843 (+0.0006 к дефолту) | 0.6982 |
| 5 | Feature engineering улучшит лучшую модель | LightGBM (дефолт и тюнинг) с FE vs без FE | 0.8841 (-0.0002 к тюнингу без FE) | 0.6985 |
| 6 | Ансамбль из топ-3 моделей превзойдёт одиночную | VotingClassifier(soft): LGBM+XGB+GB | 0.8834 (-0.0009 к одиночному) | 0.6965 |

In [ ]:
all_rows = []

for name, model in CANDIDATES.items():
    pipe = build_pipeline(model)
    pipe.fit(X_train, y_train)
    m = evaluate(pipe, X_val, y_val, split_name="val")
    all_rows.append({"model": name, "stage": "Часть 1 (дефолт)", **{k: v for k, v in m.items() if k != "split"}})

rf_m = evaluate(rf_best, X_val, y_val, split_name="val")
all_rows.append({"model": "RandomForest", "stage": "Тюнинг (Ч.2)", **{k: v for k, v in rf_m.items() if k != "split"}})

lgbm_m = evaluate(lgbm_best, X_val, y_val, split_name="val")
all_rows.append({"model": "LightGBM", "stage": "Тюнинг (Ч.2)", **{k: v for k, v in lgbm_m.items() if k != "split"}})

all_rows.append({"model": "LightGBM + FE", "stage": "Дефолт + FE (Ч.2)", **{k: v for k, v in fe_default_m.items() if k != "split"}})
all_rows.append({"model": "LightGBM + FE", "stage": "Тюнинг + FE (Ч.2)", **{k: v for k, v in fe_tuned_m.items() if k != "split"}})
all_rows.append({"model": "VotingClassifier", "stage": "Ансамбль (Ч.2)", **{k: v for k, v in ensemble_m.items() if k != "split"}})

df_all = pd.DataFrame(all_rows).sort_values("roc_auc", ascending=False)
df_all

---
# Итоговые выводы и выбор финальной модели

## Итоговая сравнительная таблица

| Подход | ROC-AUC (val) | F1-macro (val) |
|--------|--------------|----------------|
| Baseline LogReg (CP1) | 0.8536 | 0.7266 |
| DecisionTree (дефолт) | 0.6498 | 0.6491 |
| RandomForest (дефолт) | 0.8535 | 0.6716 |
| GradientBoosting (дефолт) | 0.8774 | 0.6940 |
| XGBoost (дефолт) | 0.8792 | 0.6935 |
| LightGBM (дефолт) | 0.8837 | 0.6980 |
| RandomForest (тюнинг) | 0.8766 | 0.6596 |
| LightGBM (тюнинг) | 0.8843 | 0.6982 |
| LightGBM + FE (тюнинг) | 0.8841 | 0.6985 |
| VotingClassifier (LGBM+XGB+GB) | 0.8834 | 0.6965 |

## Выводы

Эксперименты показали, что бустинговые методы (LightGBM, XGBoost, GradientBoosting) заметно превосходят DecisionTree и RandomForest уже с дефолтными параметрами. DecisionTree без ограничений набирает лишь 0.6498 из-за переобучения, тогда как LightGBM без тюнинга даёт 0.8837. LogisticRegression из CP1 оказалась сопоставима с RandomForest (0.8536 vs 0.8535), но уступает бустингу примерно на 0.03 по ROC-AUC.

Уменьшение размерности через PCA не принесло пользы: RF + PCA(10) дал 0.8536 против 0.8535 без PCA, а при большем числе компонент качество падает. Деревья умеют сами выбирать нужные признаки, поэтому смешение в PCA-компонентах только мешает.

Тюнинг дал разный эффект для двух моделей. RandomForest выиграл заметно: +0.023 (0.8535 -> 0.8766) при `max_depth=20`, `n_estimators=251`. LightGBM прибавил лишь +0.0006 (0.8837 -> 0.8843), так как его дефолтные параметры уже хорошо настроены для табличных данных.

Добавление признаков из EDA (`capital_net`, `age_group`, `is_usa`) не улучшило результат: LightGBM + FE (тюнинг) дал 0.8841 против 0.8843 без FE. Это объяснимо - градиентный бустинг самостоятельно улавливает нелинейные зависимости, из которых эти признаки и были сконструированы.

VotingClassifier из трёх бустинговых моделей также не превзошёл одиночный LightGBM (0.8834 vs 0.8843). Все три компоненты ансамбля по природе схожи, их ошибки коррелированы, поэтому усреднение вероятностей не снижает дисперсию.

## Финальная модель

Финальная модель - LightGBM с подобранными гиперпараметрами (`num_leaves=30`, `learning_rate=0.050`, `n_estimators=162`, `reg_lambda=0.199`). Среди всех протестированных подходов она показала наилучший ROC-AUC на валидации. LightGBM устойчив к переобучению за счёт регуляризации, быстро обучается и не требует нормализации признаков.

Метрики на тестовой выборке: ROC-AUC = 0.8756, F1-macro = 0.6895. Разрыв между val и test составил 0.0087, что говорит об отсутствии переобучения.

In [ ]:
candidates_final = [
    (lgbm_best,       lgbm_metrics["roc_auc"],  X_val,    X_test,    y_val,    y_test,    "LightGBM (тюнинг)"),
    (lgbm_fe_tuned,   fe_tuned_m["roc_auc"],    X_val_fe, X_test_fe, y_val_fe, y_test_fe, "LightGBM (тюнинг) + FE"),
    (ensemble,        ensemble_m["roc_auc"],     X_val,    X_test,    y_val,    y_test,    "VotingClassifier (LGBM+XGB+GB)"),
]
best = max(candidates_final, key=lambda x: x[1])
final_model, _, X_val_final, X_test_final, y_val_final, y_test_final, model_label = best

print(f"=== Финальная модель: {model_label} ===")
for split_name, X_s, y_s in [("val", X_val_final, y_val_final), ("test", X_test_final, y_test_final)]:
    m = evaluate(final_model, X_s, y_s, split_name)
    print(f"  {split_name:<6}  ROC-AUC: {m['roc_auc']:.4f}  F1-macro: {m['f1_macro']:.4f}")

save_model(final_model, BEST_MODEL_PATH)